# Notebook 0 — Environment Verification
Run this **once on the lab PC** before anything else. It checks, in order:
1. Python packages (pythonnet, ids_peak, numpy, pillow)
2. Thorlabs Kinesis DLLs load correctly
3. Config file sanity (serials filled in, offsets in range)
4. Which K10CR2 serials are visible on USB
5. Whether the IDS camera is visible

**Nothing moves.** This notebook only *looks*.

In [ ]:
# Cell 1 -- check that all required Python packages are importable
import importlib                                     # lets us try imports by name
required = ["clr",            # pythonnet: pip install pythonnet
            "ids_peak",       # IDS peak SDK python binding: pip install ids_peak
            "ids_peak_ipl",   # IDS image processing lib:    pip install ids_peak_ipl
            "numpy",          # array math:                  pip install numpy
            "PIL"]            # BMP writing:                 pip install pillow
for pkg in required:                                 # try each one
    try:
        importlib.import_module(pkg)                 # attempt the import
        print(f"  OK      {pkg}")                    # success line
    except ImportError as e:                         # missing -> tell user how to fix
        print(f"  MISSING {pkg}  ->  {e}")

In [ ]:
# Cell 2 -- can the Kinesis .NET DLLs be loaded? (mmie.motors tries at import)
import sys; sys.path.insert(0, ".")                  # make the mmie package importable
from mmie import motors                              # importing runs the DLL loading block
print("Kinesis DLLs loaded:", motors.KINESIS_OK)     # True = ready for motor control
if not motors.KINESIS_OK:                            # if False, show the exact reason
    print("Reason:", motors.KINESIS_ERR)
    print("Fix: install Thorlabs Kinesis (64-bit) and `pip install pythonnet`.")

In [ ]:
# Cell 3 -- validate the configuration file (serials, offsets, paths)
from mmie import config                              # the single source of truth
problems = config.validate_config()                  # returns [] when everything is OK
if problems:                                         # list each problem found
    for p in problems: print("  PROBLEM:", p)
else:
    print("  Config OK. Motors defined:")
    for name, m in config.MOTORS.items():            # echo the motor table for your eyes
        print(f"    {name:8s} S/N={m['serial']}  zero_offset={m['zero_offset']} deg")

In [ ]:
# Cell 4 -- scan USB for Kinesis devices (READ-ONLY, nothing moves)
if motors.KINESIS_OK:                                             # only if DLLs loaded
    from Thorlabs.MotionControl.DeviceManagerCLI import DeviceManagerCLI
    DeviceManagerCLI.BuildDeviceList()                            # scan the USB bus
    found = [str(s) for s in DeviceManagerCLI.GetDeviceList()]    # all serials present
    print("Kinesis serials on USB:", found)
    for name, m in config.MOTORS.items():                         # compare with our table
        status = "FOUND" if m["serial"] in found else "not found"
        print(f"  {name:8s} (S/N {m['serial']}): {status}")

In [ ]:
# Cell 5 -- is the IDS camera visible? (READ-ONLY, no acquisition)
from mmie import camera                              # importing tests the ids_peak import
print("IDS peak importable:", camera.PEAK_OK)
if camera.PEAK_OK:
    from ids_peak import ids_peak                    # list cameras without opening them
    ids_peak.Library.Initialize()                    # start runtime
    dm = ids_peak.DeviceManager.Instance()           # device manager singleton
    dm.Update()                                      # scan
    for d in dm.Devices():                           # print every camera found
        print("  Camera:", d.ModelName(), "| S/N", d.SerialNumber())
    if dm.Devices().empty(): print("  No IDS camera detected -- check USB3 cable.")
    ids_peak.Library.Close()                         # release runtime
else:
    print("Reason:", camera.PEAK_ERR)
    print("Fix: install IDS peak from ids-imaging.com, then `pip install ids_peak ids_peak_ipl`.")